# Compare GA and ClustalW

In [1]:
# Import our algorithms
%run ga.ipynb

# Import Biopython for ClustalW
from Bio import SeqIO, AlignIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import subprocess
import time
import os

## Helper Functions

In [2]:
def run_clustalw(sequences, output_prefix="temp"):
    clustalw_path = r"C:\Program Files (x86)\ClustalW2\clustalw2.exe"
    
    fasta_file = f"{output_prefix}.fasta"
    records = [SeqRecord(Seq(seq), id=f"seq{i}", description="") 
               for i, seq in enumerate(sequences)]
    SeqIO.write(records, fasta_file, "fasta")
    
    # Run ClustalW using subprocess
    start = time.time()
    try:
        cmd = [
            clustalw_path,
            f"-INFILE={fasta_file}",
            "-TYPE=PROTEIN",
            "-MATRIX=BLOSUM",      # Uses BLOSUM62
            "-GAPOPEN=10",         # Gap open penalty 
            "-GAPEXT=1"            # Gap extend penalty 
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        elapsed = time.time() - start
        
        if result.returncode != 0:
            print(f"ClustalW error: {result.stderr}")
            return None, elapsed
    except FileNotFoundError:
        print(f"Error: clustalw2 not found at {clustalw_path}")
        print("Update the clustalw_path variable if installed elsewhere")
        return None, 0
    except Exception as e:
        print(f"Error running ClustalW: {e}")
        return None, 0
    
    # Read alignment
    aln_file = f"{output_prefix}.aln"
    try:
        alignment = AlignIO.read(aln_file, "clustal")
        aligned_seqs = [str(record.seq) for record in alignment]
    except Exception as e:
        print(f"Error reading alignment: {e}")
        return None, elapsed
    
    for ext in [".fasta", ".aln", ".dnd"]:
        try:
            os.remove(f"{output_prefix}{ext}")
        except:
            pass
    
    return aligned_seqs, elapsed

In [3]:
def compare_algorithms(sequences, max_gaps=3, ga_params=None):
    if ga_params is None:
        ga_params = {
            'population_size': 100,
            'num_generations': 50,
            'tournament_size': 10,
            'mutation_prob': 0.1,
            'elitism_size': 10
        }
    
    print(f"Sequences: {sequences}")
    print(f"Max gaps: {max_gaps}")
    print()
    
    # Run GA
    print("=== GENETIC ALGORITHM ===")
    start = time.time()
    ga_individual, _ = ga_msa(sequences, max_gaps=max_gaps, **ga_params)
    ga_time = time.time() - start
    ga_score = ga_individual.fitness
    print(f"Score: {ga_score:.2f}, Time: {ga_time:.3f}s")
    for seq in ga_individual.code:
        print(f"  {seq}")
    print()
    
    # Run ClustalW
    print("=== CLUSTALW ===")
    clustalw_alignment, clustalw_time = run_clustalw(sequences)
    if clustalw_alignment:
        clustalw_score = calculate_sp_score(clustalw_alignment)
        print(f"Score: {clustalw_score:.2f}, Time: {clustalw_time:.3f}s")
        for seq in clustalw_alignment:
            print(f"  {seq}")
    else:
        clustalw_score = None
        print("ClustalW failed to run")
    print()
    
    # Summary
    print("=== SUMMARY ===")
    print(f"GA:          Score={ga_score:.2f}, Time={ga_time:.3f}s")
    if clustalw_score:
        print(f"ClustalW:    Score={clustalw_score:.2f}, Time={clustalw_time:.3f}s")
        print(f"Difference:  {ga_score - clustalw_score:.2f} (GA - ClustalW)")
    print("="*60)
    
    return {
        'sequences': sequences,
        'ga_score': ga_score,
        'ga_time': ga_time,
        'clustalw_score': clustalw_score,
        'clustalw_time': clustalw_time
    }

## Batch Testing

In [4]:
test_cases = [
    # Original test cases
    (['ACT', 'ACT', 'ACT'], 3),
    (['ACT', 'AT', 'ACT'], 3),
    (['AG', 'A', 'AG'], 3),
    (['ACGT', 'AGT', 'AT'], 3),
    (['GAT', 'AT', 'T'], 3),
    (['GATTACA', 'GTACA', 'GACA'], 3),
    (['ATCG', 'ACG', 'ATG'], 3),
    (['CGTA', 'CTA', 'CGA'], 3),
    (['ACE', 'AE', 'CE'], 3),
    (['PAWHE', 'HEAWY', 'AWHE'], 3),
    (['ACDEG', 'ACEG', 'ADEG'], 3),
    (['PAWH', 'AWH', 'PWH', 'AH'], 3),
    (['ACGT', 'AGT'], 3),
    (['HELIX', 'HEAX'], 3),
    (['A', 'C', 'T'], 3),
    (['W', 'H', 'E'], 3),
    (['ATGATG', 'ATGTG', 'ATG'], 3),
    (['ABCD', 'BCD', 'CD', 'D'], 3),
    (['GATTACAGAT', 'GATTGAT', 'GACAGAT', 'GATTAGAT'], 3),
    (['HELICASE', 'HELCASE', 'HECASE', 'HEASE'], 3),
    
    # BAliBASE RV11 Test Cases
    
    # BB11001 - Homeodomain proteins 
    (['GKGDPKKPRGKMSSYAFFVQTSREEHKKKHPDASVNFSEFSKKCSERWKTMSAKEKGKFEDMAKADKARYEREMKTYIPPKGE',
      'MQDRVKRPMNAFIVWSRDQRRKMALENPRMRNSEISKQLGYQWKMLTEAEKWPFFQEAQKLQAMHREKYPNYKYRPRRKAKMLPK',
      'MKKLKKHPDFPKKPLTPYFRFFMEKRAKYAKLHPEMSNLDLTKILSKKYKELPEKKKMKYIQDFQREKQEFERNLARFREDHPDLIQNAKK',
      'MHIKKPLNAFMLYMKEMRANVVAESTLKESAAINQILGRRWHALSREEQAKYYELARKERQLHMQLYPGWSARDNYGKKKKRKREK'], 10),
    
    # BB11002 - Helix-turn-helix 
    (['NLFVALYDFVASGDNTLSITKGEKLRVLGYNHNGEWCEAQTKNGQGWVPSNYITPVNS',
      'PLALLLDSSLEGEFDLVQRIIYEVDDPSLPNDEGITALHNAVCAGHTEIVKFLVQFGVNVNAADSDGWTPLHCAASCNNVQVCKFLVESGAAVFAMTYSDMQTAADKCEEMEEGYTQCSQFLYGVQEKMGIMNKGVIYALWDYEPQNDDELPMKEGDCMTIIHREDEDEIEWWWARLNDKEGYVPRNLLGLYP',
      'AEGYQYRALYDYKKEREEDIDLHLGDILTVNKGSLVALGFSDGQEARPEEIGWLNGYNETTGERGDFPGTYVEYIGRKKISPP',
      'MIQNFRVYYRDSRDPVWKGPAKLLWKGEGAVVIQDNSDIKVVPRRKAKIIRD',
      'TTGRLDLPPGFMFKVQAQHDYTATDTDELQLKAGDVVLVIPFQNPEEQDEGWLMGVKESDWNQHKELEKCRGVFPENFTERVQ',
      'ADRKLCADQECSHPISMAVALQDYMAPDCRFLTIHRGQVVYVFSKLKGRGRLFWGGSVQGDYYGDLAARLGYFPSSIVREDQTLKPGKVDVKTDKWDFYCQ',
      'LGSPEFIILQTYRAIADYEKTSGSEMALSTGDVVEVVEKSESGWWFCQMKAKRGWIPASFLEPLDSPDETEDPEPNYAGEPYVAIKAYTAVEGDEVSLLEGEAVEVIHKLLDGWWVIRKDDVTGYFPSMYLQKS',
      'GSSGSSGEIAQVTSAYVASGSEQLSLAPGQLILILKKNTSGWWQGELQARGKKRQKGWFPASHVKLLGPSSERASGPSSG'], 15),
    
    # BB11003 - DNA/RNA-binding proteins
    (['SISDTVKRAREAFNSGKTRSLQFRIQQLEALQRMINENLKSISGALASDLGKNEWTSYYEEVAHVLEELDTTIKELPDWAEDEPVAKTRQTQQDDLYIHSEPLGVVLVIGAWNYPFNLTIQPMVGAVAAGNAVILKPSEVSGHMADLLATLIPQYMDQNLYLVVKGGVPETTELLKERFDHIMYTGSTAVGKIVMAAAAKHLTPVTLELGGKSPCYVDKDCDLDVACRRIAWGKFMNSGQTCVAPDYILCDPSIQNQIVEKLKKSLKDFYGEDAKQSRDYGRIINDRHFQRVKGLIDNQKVAHGGTWDQSSRYIAPTILVDVDPQSPVMQEEIFGPVMPIVCVRSLEEAIQFINQREKPLALYVFSNNEKVIKKMIAETSSGGVTANDVIVHITVPTLPFGGVGNSGMGAYHGKKSFETFSHRRSCLVKSLLNEEAHKARYPPSPA',
      'MTVEPFRNEPIETFQTEEARRAMREALRRVREEFGRHYPLYIGGEWVDTKERMVSLNPSAPSEVVGTTAKAGKAEAEAALEAAWKAFKTWKDWPQEDRSRLLLKAAALMRRRKRELEATLVYEVGKNWVEASADVAEAIDFIEYYARAALRYRYPAVEVVPYPGEDNESFYVPLGAGVVIAPWNFPVAIFTGMIVGPVAVGNTVIAKPAEDAVVVGAKVFEIFHEAGFPPGVVNFLPGVGEEVGAYLVEHPRIRFINFTGSLEVGLKIYEAAGRLAPGQTWFKRAYVETGGKNAIIVDETADFDLAAEGVVVSAYGFQGQKCSAASRLILTQGAYEPVLERVLKRAERLSVGPAEENPDLGPVVSAEQERKVLSYIEIGKNEGQLVLGGKRLEGEGYFIAPTVFTEVPPKARIAQEEIFGPVLSVIRVKDFAEALEVANDTPYGLTGGVYSRKREHLEWARREFHVGNLYFNRKITGALVGVQPFGGFKLSGTNAKTGALDYLRLFLEMKAVAERF',
      'TDNVFYATNAFTGEALPLAFPVHTEVEVNQAATAAAKVARDFRRLNNSKRASLLRTIASELEARSDDIIARAHLETALPEVRLTGEIARTANQLRLFADVVNSGSYHQAILDTPNPTRAPLPKPDIRRQQIALGPVAVFGASNFPLAFSAAGGDTASALAAGCPVIVKGHTAHPGTSQIVAECIEQALKQEQLPQAIFTLLQGNQRALGQALVSHPEIKAVGFTGSVGGGRALFNLAHERPEPIPFYGELGAINPTFIFPSAMRAKADLADQFVASMTMGCGQFCTKPGVVFALNTPETQAFIETAQSLIRQQSPSTLLTPGIRDSYQSQVVSRGSDDGIDVTFSQAESPCVASALFVTSSENWRKHPAWEEEIFGPQSLIVVCENVADMLSLSEMLAGSLTATIHATEEDYPQVSQLIPRLEEIAGRLVFNGWPTGVEVGYAMVHGGPYPASTHSASTSVGAEAIHRWLRPVAYQALPESLLPDSLKAENPLEIARAVDGKAA',
      'DELLEKAKKVREAWDVLRNATTREKNKAIKKIAEKLDERRKEILEANRIDVEKARERGVKESLVDRLALNDKRIDEXIKACETVIGLKDPVGEVIDSWVREDGLRIARVRVPIGPIGIIYESRPNVTVETTILALKSGNTILLRGGSDALNSNKAIVSAIREALKETEIPESSVEFIENTDRSLVLEXIRLREYLSLVIPRGGYGLISFVRDNATVPVLETGVGNCHIFVDESADLKKAVPVIINAKTQRPGTCNAAEKLLVHEKIAKEFLPVIVEELRKHGVEVRGCEKTREIVPDVVPATEDDWPTEYLDLIIAIKVVKNVDEAIEHIKKYSTGHSESILTENYSNAKKFVSEIDAAAVYVNASTRFTDGGQFGFGAEIGISTQRFHARGPVGLRELTTYKFVVLGEYHVRE'], 15),
    
    # BB11008 - Mixed proteins
    (['LQDAEWYWGDISREEVNEKLRDTADGTFLVRDASTKMHGDYTLTLRKGGNNKLIKIFHRDGKYGFSDPLTFNSVVELINHYRNESLAQYNPKLDVKLLYPVSKY',
      'GSPASGTSLSAAIHRTQLWFHGRISREESQRLIGQQGLVDGLFLVRESQRNPQGFVLSLCHLQKVKHYLILPSEEEGRLYFSMDDGQTRFTDLLQLVEFHQLNRGILPCLLRHCCTRVAL',
      'SSPQPILDTIYKLLSEQEQTLVQMIHEQSLLLNRLPPTLDENSLAPLKSLSQKQITLSGQMNTEMSALDATKKGMILEPTDLAKLFALKQDLQIQFKQLSLLHNEIQSILNPQHSAPKPNVALVLKSQPFPVVISKGKQLGENQLVVLVLTGARSNFHINGPVKATMICDSHPPTTPLEMDSQPIYPATLTAHFPLKFLAGTRKCSVNLKFGVNIRDLDNVTTTVESDASNPFVVITNECQWEGSAGVLLKKDAFDGQLEITWAQFINTLQRHFLIATKQDPVRPKRPLSSYDLKYIQTHFFGNRSIIHQQDFDKFWVWFGKSMQTLRYQRHISTLWQEGIIYGYMGRQEVNDALQNQDPGTFIIRFSERNPGQFGIAYIGVEMPARIKHYLVQPNDTAAAKKTFPDFLSEHSQFVNLLQWTKDTNGAPRFLKLHKDTALGSFAPKRTAPVPVGGX',
      'LDKQKELDSKVRNVKDKVMCIEHEIKSLEDLQDEYDFKCKTLQNREHLLLKKMYLMLDNKRKEVVHKIIELLNVTELTQNALINDELVEWKRRQQSACIGGPPNACLDQLQNWFTIVAESLQQVRQQLKKLEELEQKYTYEHDPITKNKQVLWDRTFSLFQQLIQSSFVVERQPCMPTHPQRPLVLKTGVQFTVKLRLLVKLQELNYNLKVKVLFDKDVNERNTVKGFRKFNILGTHTKVMNMEESTNGSLAAEFRHLQLKEQKNAGTRTNEGPLIVTEELHSLSFETQLCQPGLVIDLETTSLPVVVISNVSQLPSGWASILWYNMLVAEPRNLSFFLTPPCARWAQLSEVLSWQFSSVTKRGLNVDQLNMLGEKLLGPNASPDGLIPWTRFCKENINDKNFPFWLWIESILELIKKHLLPLWNDGCIMGFISKERERALLKDQQPGTFLLRFSESSREGAITFTWVERSQNGGEPDFHAVEPYTKKELSAVTFPDIIRNYKVMAAENIPENPLKYLYPNIDKDHAFGKYYSRGXIKTE'], 20),
]

results = []
for i, (seqs, max_gaps) in enumerate(test_cases, 1):
    print(f"\n{'='*60}")
    print(f"TEST {i}/{len(test_cases)}")
    print(f"{'='*60}\n")
    result = compare_algorithms(seqs, max_gaps=max_gaps)
    results.append(result)
    print("\n")


TEST 1/24

Sequences: ['ACT', 'ACT', 'ACT']
Max gaps: 3

=== GENETIC ALGORITHM ===
Score: 54.00, Time: 0.262s
  ACT---
  ACT---
  ACT---

=== CLUSTALW ===
Score: 54.00, Time: 0.021s
  ACT
  ACT
  ACT

=== SUMMARY ===
GA:          Score=54.00, Time=0.262s
ClustalW:    Score=54.00, Time=0.021s
Difference:  0.00 (GA - ClustalW)



TEST 2/24

Sequences: ['ACT', 'AT', 'ACT']
Max gaps: 3

=== GENETIC ALGORITHM ===
Score: 16.00, Time: 0.174s
  ACT--
  A-T--
  ACT--

=== CLUSTALW ===
Score: 8.00, Time: 0.012s
  ACT
  ACT
  -AT

=== SUMMARY ===
GA:          Score=16.00, Time=0.174s
ClustalW:    Score=8.00, Time=0.012s
Difference:  8.00 (GA - ClustalW)



TEST 3/24

Sequences: ['AG', 'A', 'AG']
Max gaps: 3

=== GENETIC ALGORITHM ===
Score: -2.00, Time: 0.221s
  A-G-
  A---
  A-G-

=== CLUSTALW ===
Score: -2.00, Time: 0.012s
  AG
  AG
  A-

=== SUMMARY ===
GA:          Score=-2.00, Time=0.221s
ClustalW:    Score=-2.00, Time=0.012s
Difference:  0.00 (GA - ClustalW)



TEST 4/24

Sequences: ['ACGT

## Results Summary

In [5]:
print("\n" + "="*80)
print("OVERALL SUMMARY")
print("="*80)
print(f"{'Test':<5} {'Sequences':<30} {'GA Score':>10} {'CW Score':>10} {'GA Time':>10} {'CW Time':>10}")
print("-"*80)

for i, r in enumerate(results, 1):
    seqs_str = str(r['sequences'])[:30]
    cw_score = r['clustalw_score'] if r['clustalw_score'] else float('nan')
    cw_time = r['clustalw_time'] if r['clustalw_time'] else float('nan')
    print(f"{i:<5} {seqs_str:<30} {r['ga_score']:>10.2f} {cw_score:>10.2f} {r['ga_time']:>9.3f}s {cw_time:>9.3f}s")

print("-"*80)

# Calculate statistics
total_tests = len([r for r in results if r['clustalw_score'] is not None])
ga_better = sum(1 for r in results if r['clustalw_score'] and r['ga_score'] > r['clustalw_score'])
cw_better = sum(1 for r in results if r['clustalw_score'] and r['clustalw_score'] > r['ga_score'])
tied = sum(1 for r in results if r['clustalw_score'] and abs(r['ga_score'] - r['clustalw_score']) < 0.01)

avg_ga_time = sum(r['ga_time'] for r in results) / len(results)
avg_cw_time = sum(r['clustalw_time'] for r in results if r['clustalw_time']) / total_tests

print(f"\nScore Comparison:")
print(f"  GA better: {ga_better}/{total_tests} ({ga_better/total_tests*100:.1f}%)")
print(f"  ClustalW better: {cw_better}/{total_tests} ({cw_better/total_tests*100:.1f}%)")
print(f"  Tied: {tied}/{total_tests} ({tied/total_tests*100:.1f}%)")
print(f"\nAverage Time:")
print(f"  GA: {avg_ga_time:.3f}s")
print(f"  ClustalW: {avg_cw_time:.3f}s")


OVERALL SUMMARY
Test  Sequences                        GA Score   CW Score    GA Time    CW Time
--------------------------------------------------------------------------------
1     ['ACT', 'ACT', 'ACT']               54.00      54.00     0.262s     0.021s
2     ['ACT', 'AT', 'ACT']                16.00       8.00     0.174s     0.012s
3     ['AG', 'A', 'AG']                   -2.00      -2.00     0.221s     0.012s
4     ['ACGT', 'AGT', 'AT']                2.00     -10.00     0.192s     0.018s
5     ['GAT', 'AT', 'T']                 -12.00     -12.00     0.396s     0.014s
6     ['GATTACA', 'GTACA', 'GACA']        41.00      17.00     0.226s     0.017s
7     ['ATCG', 'ACG', 'ATG']              17.00       9.00     0.196s     0.012s
8     ['CGTA', 'CTA', 'CGA']              21.00      20.00     0.189s     0.015s
9     ['ACE', 'AE', 'CE']                  4.00       4.00     0.173s     0.012s
10    ['PAWHE', 'HEAWY', 'AWHE']          10.00      10.00     0.194s     0.012s
11    ['ACD